# Course Recommendation System

---
### Import necessary libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

---
### Data Loading, Cleaning, and Title-Based Deduplication

In [2]:
# Load the dataset
df = pd.read_csv('course_data.csv')

# Basic Data Inspection
print(df.head())
print(df.info())

# FIX: Drop duplicates based on title instead of course_id
# This prevents dropping unique courses (like Figma) that share a duplicate ID (C049) by mistake
df = df.drop_duplicates(subset='course_title')
df = df.reset_index(drop=True)

  course_id                   course_title         category   instructor  \
0      C016      Cloud Architecture Basics  Cloud Computing    Mike Ross   
1      C017     Cybersecurity Fundamentals    Cybersecurity   Emily Chen   
2      C018     Digital Marketing Strategy        Marketing     David Wu   
3      C019  Project Management Essentials         Business  Linda Zhang   
4      C020             Python for Finance     Data Science     John Doe   

   rating difficulty_level  students_enrolled  
0     4.3         Beginner               1200  
1     4.6         Beginner               2200  
2     4.2     Intermediate               1500  
3     4.4         Beginner               2800  
4     4.7         Advanced                950  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   course_id          100 non-null    object 
 1   cou

In [3]:
# Clean categorical text fields to prevent token splitting (e.g., "Mike Ross" -> "mikeross")
def clean_data(x):
    if isinstance(x, str):
        return x.replace(" ", "").lower()
    return ""

# Apply cleaning to categorical columns
for col in ['category', 'instructor', 'difficulty_level']:
    df[col + '_cleaned'] = df[col].apply(clean_data)

# Combine features with clean categorical strings and a lowercased title
def combine_features(row):
    return f"{row['course_title'].lower()} {row['category_cleaned']} {row['instructor_cleaned']} {row['difficulty_level_cleaned']}"

df['combined_features'] = df.apply(combine_features, axis=1)

print(df[['course_title', 'combined_features']].head())

                    course_title  \
0      Cloud Architecture Basics   
1     Cybersecurity Fundamentals   
2     Digital Marketing Strategy   
3  Project Management Essentials   
4             Python for Finance   

                                   combined_features  
0  cloud architecture basics cloudcomputing miker...  
1  cybersecurity fundamentals cybersecurity emily...  
2  digital marketing strategy marketing davidwu i...  
3  project management essentials business lindazh...  
4    python for finance datascience johndoe advanced  


---
### Text Vectorization and Pairwise Cosine Similarity Calculation
We convert the corpus of `combined_features` text profiles into a Bag-of-Words matrix using text counts, then map their spatial relationships using cosine distance. 

*Note: Since short course titles share frequent stopwords (e.g., "for", "and", "basics"), keeping text domains structural (by combining standard tags with custom space stripping) ensures matching remains highly specific to identical categories or instructors.*

In [4]:
# Convert the combined features into a matrix of token counts
cv = CountVectorizer()
count_matrix = cv.fit_transform(df['combined_features'])

# Compute the Cosine Similarity based on the count_matrix
cosine_sim = cosine_similarity(count_matrix)

print(f"Similarity Matrix Shape: {cosine_sim.shape}")

Similarity Matrix Shape: (100, 100)


---
### Recommender Logic

In [5]:
def get_recommendations(course_title, cosine_sim_matrix=cosine_sim, data=df, top_n=5):
    # FIX: Use case-insensitive matching to handle varied user inputs safely
    try:
        idx = data[data['course_title'].str.lower() == course_title.lower()].index[0]
    except IndexError:
        return "Course not found."

    # Get pairwise similarity scores
    sim_scores = list(enumerate(cosine_sim_matrix[idx]))

    # Filter out the target course index to prevent self-recommendation
    sim_scores = [item for item in sim_scores if item[0] != idx]

    # Sort the courses based on similarity scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the scores of the top_n most similar courses
    sim_scores = sim_scores[:top_n]

    # Get the course indices
    course_indices = [i[0] for i in sim_scores]

    return data[['course_title', 'category', 'difficulty_level']].iloc[course_indices]

# Test the recommender with lowercase input to verify flexibility
print(get_recommendations("cloud architecture basics"))

                  course_title         category difficulty_level
56  Microservices Architecture  Cloud Computing         Advanced
14        Azure Cloud Services  Cloud Computing     Intermediate
21       Google Cloud Platform  Cloud Computing     Intermediate
70      Multi-Cloud Strategies  Cloud Computing         Advanced
91     Cloud Cost Optimization  Cloud Computing     Intermediate


---
### Save the similarity matrix and the dataframe for production use

In [6]:
model_data = {
    'similarity_matrix': cosine_sim,
    'data': df
}

with open('model.pkl', 'wb') as file:
    pickle.dump(model_data, file)

print("Model saved successfully as model.pkl")

Model saved successfully as model.pkl


---
### Verify Production Model Artifact

In [7]:
# Load the saved model data to verify integrity
with open('model.pkl', 'rb') as file:
    loaded_model = pickle.load(file)

loaded_sim = loaded_model['similarity_matrix']
loaded_df = loaded_model['data']

print("Loaded Model Verification Success!")
print("\nTesting pipeline with loaded production assets:")
print(get_recommendations("Python for Finance", cosine_sim_matrix=loaded_sim, data=loaded_df))

Loaded Model Verification Success!

Testing pipeline with loaded production assets:
                course_title      category difficulty_level
74    Reinforcement Learning  Data Science         Advanced
46   Time Series Forecasting  Data Science         Advanced
32  Tableau for Data Science  Data Science         Beginner
18      Statistical Analysis  Data Science     Intermediate
60       Bayesian Statistics  Data Science     Intermediate
